In [1]:
import pandas as pd
import numpy as np
import requests
import urllib.request
import json
import geopandas as gpd
from shapely.wkt import loads

In [2]:
key = '01e50810998b0acda3ea2cba960a804ac68205ea'

In [3]:
baseUrl = "https://api.census.gov/data/2021/acs/acs5"
API_key = f'&key={key}'

In [4]:
# B19025_001E household income
# B25003_001E number of households
# B25003_002E number of owner households

In [5]:
# define URL for the Data Sets endpoint 
variables_bg = "?get=B19025_001E,B25003_001E,B25003_002E"
geo_bg = "&for=block%20group:*&in=county:019&in=state:45"

# open the URL as defined above and create a the request object 
request_1 = urllib.request.urlopen(baseUrl + variables_bg + geo_bg + API_key)

# actually read the data
result_variables_bg = request_1.read()

In [6]:
# define URL for the Data Sets endpoint 
variables_tract = "?get=B25120_002E,B25003_002E"
geo_tract = "&for=tract:*&in=county:019&in=state:45"

# open the URL as defined above and create a the request object 
request_2 = urllib.request.urlopen(baseUrl + variables_tract + geo_tract + API_key)

# actually read the data
result_variables_tract = request_2.read()

In [7]:
# transform to Python dictionary
jsonData_bg = json.loads(result_variables_bg.decode('utf-8'))
df_bg = pd.DataFrame(jsonData_bg)

In [8]:
# transform to Python dictionary
jsonData_tract = json.loads(result_variables_tract.decode('utf-8'))
df_tract = pd.DataFrame(jsonData_tract)

In [9]:
# first row are the column names, this needs to be fixed
df_tract.columns = df_tract.iloc[0]
df_tract = df_tract.iloc[1:]

In [10]:
# first row are the column names, this needs to be fixed
df_bg.columns = df_bg.iloc[0]
df_bg = df_bg.iloc[1:]

In [11]:
df_tract = df_tract.rename(columns={'B25003_002E': 'owner_households_tract', 'B25120_002E': 'total_owner_i_0_tract'})
df_bg = df_bg.rename(columns={'B19025_001E': 'i_0','B25003_001E':'#_households','B25003_002E':'owner_households'})

In [12]:
merged_df = pd.merge(df_bg, df_tract[['state', 'county', 'tract', 'total_owner_i_0_tract', 'owner_households_tract']], on=['state', 'county', 'tract'], how='left')
merged_df['GEOID'] = merged_df["state"] + merged_df["county"] + merged_df["tract"] + merged_df['block group']
merged_df

,i_0,#_households,owner_households,state,county,tract,block group,total_owner_i_0_tract,owner_households_tract,GEOID
0,42430300,303,165,45,019,000100,1,160095300,664,450190001001
1,63660700,316,254,45,019,000100,2,160095300,664,450190001002
2,68497300,262,245,45,019,000100,3,160095300,664,450190001003
3,55171400,243,195,45,019,000200,1,165931300,506,450190002001
4,126362700,366,311,45,019,000200,2,165931300,506,450190002002
...,...,...,...,...,...,...,...,...,...,...
256,201103800,1786,1088,45,019,005800,3,262790600,2096,450190058003
257,72364500,519,369,45,019,005900,1,166856500,1139,450190059001
258,39654400,406,237,45,019,005900,2,166856500,1139,450190059002
259,98738400,844,533,45,019,005900,3,166856500,1139,450190059003


In [13]:
merged_df = merged_df.apply(pd.to_numeric, errors='coerce')

In [14]:
merged_df['fraction_owner_households'] = merged_df['owner_households']/merged_df['#_households']
merged_df['owner_i_0_tract'] = merged_df['total_owner_i_0_tract']/merged_df['owner_households_tract']

In [15]:
merged_df.head(2)

,i_0,#_households,owner_households,state,county,tract,block group,total_owner_i_0_tract,owner_households_tract,GEOID,fraction_owner_households,owner_i_0_tract
0,42430300.0,303,165,45,19,100,1,160095300.0,664,450190001001,0.544554,241107.379518
1,63660700.0,316,254,45,19,100,2,160095300.0,664,450190001002,0.803797,241107.379518


In [16]:
# columns_to_drop = ['owner_households_tract', 'total_owner_i_0_tract',"state", "county", "tract", "block group"]
# df = merged_df.drop(columns=columns_to_drop)

## load in shapefiles

### Check on NaN values and other issues

In [19]:
# check for block groups with no owner-occupied households
merged_df[merged_df['fraction_owner_households'] <= 0]

,i_0,#_households,owner_households,state,county,tract,block group,total_owner_i_0_tract,owner_households_tract,GEOID,fraction_owner_households,owner_i_0_tract
87,26100400.0,447,0,45,19,2612,2,127006400.0,930,450190026122,0.0,1.365660e+05
132,10836500.0,283,0,45,19,3111,2,56076900.0,910,450190031112,0.0,6.162297e+04
140,30435500.0,704,0,45,19,3116,1,52327100.0,521,450190031161,0.0,1.004359e+05
145,12449600.0,204,0,45,19,3200,1,-666666666.0,0,450190032001,0.0,-inf
146,14323800.0,192,0,45,19,3200,2,-666666666.0,0,450190032002,0.0,-inf
238,4864400.0,289,0,45,19,5300,3,70586700.0,401,450190053003,0.0,1.760267e+05


In [20]:
# drop the block groups with no owner-occupied households
owners_df = merged_df.loc[merged_df['fraction_owner_households'] > 0]

In [21]:
owners_df['ave_i_0'] = owners_df['i_0']/owners_df['#_households']

/var/folders/3_/tq4vdvfj38q890xp77mccl040000gn/T/ipykernel_19675/811127925.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  owners_df['ave_i_0'] = owners_df['i_0']/owners_df['#_households']


In [22]:
# replace nan value with tract level owner income
owners_df['ave_i_0'] = owners_df['ave_i_0'].fillna(owners_df['owner_i_0_tract'])

/var/folders/3_/tq4vdvfj38q890xp77mccl040000gn/T/ipykernel_19675/645195007.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  owners_df['ave_i_0'] = owners_df['ave_i_0'].fillna(owners_df['owner_i_0_tract'])


In [23]:
nan_count = owners_df['i_0'].isna().sum()

print(f"The 'ave_i_0' column has {nan_count} NaN values.")

The 'ave_i_0' column has 1 NaN values.


In [24]:
nan_count = owners_df['ave_i_0'].isna().sum()

print(f"The 'ave_i_0' column has {nan_count} NaN values.")

The 'ave_i_0' column has 0 NaN values.


In [25]:
columns_to_drop = ['owner_households_tract', 'total_owner_i_0_tract',"state", "county", "tract", "block group",'owner_households','i_0','fraction_owner_households','owner_i_0_tract']
df = owners_df.drop(columns=columns_to_drop)

In [26]:
# df_damage_clean = pd.read_csv('data/damage_shape_hurricane.csv')
df_damage = pd.read_csv('data/damage_shape_bigfile.csv')
df_damage_hurricane = pd.read_csv('data/damage_shape_bigfile.csv')

# merge with shapefile
complete_df = df.merge(df_damage, on = "GEOID")
complete_df_hurricane = df.merge(df_damage_hurricane, on = "GEOID")

In [27]:
complete_df.to_csv('data/census_damage_data.csv', index=False)
complete_df_hurricane.to_csv('data/census_damage_data_hurricane.csv', index=False)

In [28]:
complete_df_hurricane.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 7 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   #_households                     246 non-null    int64  
 1   GEOID                            246 non-null    int64  
 2   ave_i_0                          246 non-null    float64
 3   geometry                         246 non-null    object 
 4   Max Potential Damage: Structure  246 non-null    float64
 5   Damage: Structure                246 non-null    float64
 6   Risk (EAD)                       246 non-null    float64
dtypes: float64(4), int64(2), object(1)
memory usage: 13.6+ KB
